In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import os
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer

In [2]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/himau23/preprocessed-question-pairs/preprossed(1).csv
/kaggle/input/datasets/quora/question-pairs-dataset/questions.csv


In [3]:
DATA_PATH = "/kaggle/input/datasets/quora/question-pairs-dataset/questions.csv"
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (404351, 6)
Columns: ['id', 'qid1', 'qid2', 'question1', 'question2', 'is_duplicate']


,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [4]:
MODEL_NAME = "BAAI/bge-small-en-v1.5"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)
model.eval()

print("Device:", device)

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Device: cuda


In [5]:
def get_embedding(text):
    encoded = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )
    
    encoded = {
        key: value.to(device)
        for key, value in encoded.items()
    }
    
    with torch.no_grad():
        outputs = model(**encoded)
        
    token_embeddings = outputs.last_hidden_state
    attention_mask = encoded["attention_mask"]
    
    mask = attention_mask.unsqueeze(-1).expand(
        token_embeddings.size()
    ).float()
    
    embedding = torch.sum(
        token_embeddings * mask,
        dim=1
    ) / torch.clamp(
        mask.sum(dim=1),
        min=1e-9
    )
    
    embedding = F.normalize(
        embedding,
        p=2,
        dim=1
    )
    
    return embedding

In [6]:
q1 = df.loc[0, "question1"]
q2 = df.loc[0, "question2"]

print("Question 1:", q1)
print("Question 2:", q2)
print("Label:", df.loc[0, "is_duplicate"])

e1 = get_embedding(q1)
e2 = get_embedding(q2)

print("Embedding 1:", e1.shape)
print("Embedding 2:", e2.shape)

similarity = torch.sum(e1 * e2).item()
print("BGE Semantic Similarity:", similarity)

Question 1: What is the step by step guide to invest in share market in india?
Question 2: What is the step by step guide to invest in share market?
Label: 0
Embedding 1: torch.Size([1, 384])
Embedding 2: torch.Size([1, 384])
BGE Semantic Similarity: 0.920744776725769


In [7]:
bge_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5",
    device="cuda"
)

questions1 = df["question1"].fillna("").astype(str).tolist()
questions2 = df["question2"].fillna("").astype(str).tolist()

embeddings_q1 = bge_model.encode(
    questions1,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings_q2 = bge_model.encode(
    questions2,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/6318 [00:00<?, ?it/s]

Batches:   0%|          | 0/6318 [00:00<?, ?it/s]

In [8]:
bge_similarity = np.sum(
    embeddings_q1 * embeddings_q2,
    axis=1
)

df["bge_similarity"] = bge_similarity
print("Q1 embeddings:", embeddings_q1.shape)
print("Q2 embeddings:", embeddings_q2.shape)
df[["question1", "question2", "is_duplicate", "bge_similarity"]].head(10)

Q1 embeddings: (404351, 384)
Q2 embeddings: (404351, 384)


,question1,question2,is_duplicate,bge_similarity
0,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0,0.922287
1,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0,0.693393
2,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0,0.826137
3,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0,0.505303
4,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0,0.552441
5,Astrology: I am a Capricorn Sun Cap moon and c...,"I'm a triple Capricorn (Sun, Moon and ascendan...",1,0.858212
6,Should I buy tiago?,What keeps childern active and far from phone ...,0,0.434120
7,How can I be a good geologist?,What should I do to be a great geologist?,1,0.928560
8,When do you use シ instead of し?,"When do you use ""&"" instead of ""and""?",0,0.700380
9,Motorola (company): Can I hack my Charter Moto...,How do I hack Motorola DCX3400 for free internet?,0,0.835582


In [9]:
df.to_csv("/kaggle/working/embedded_questions_quora.csv", index=False)